The script serves as the Data Generation Layer of the MVP pipeline. It programmatically simulates 5,000 historical ride-sharing event logs over a continuous 3-week timeline leading up to the current execution date. This replaces the need for complex, live webhook data streams while maintaining structural integrity for downstream ETL processes in Google BigQuery.

The script utilizes two primary Python libraries built for data manipulation and numerical generation:
* **Pandas (pd):** Used to construct the primary data structure (DataFrame), format data types, sort records sequentially, and handle the file I/O operations to export the clean CSV.
* **Numpy (np):** Leveraged for random sampling, vectorization, and generating mock metrics using statistical distributions (normal distributions for fares and travel durations).

#### The 3-Week Timeline
**Uniform Randomization:**
Generates an array of random integers between 0 and the maximum minutes in 3 weeks (30,240). Each trip is assigned a randomized timestamp by shifting the `start_date` forward by that number of minutes.

#### Geospatial & Operational Logic
**Weighted Geography:**
Districts are sampled with unequal probabilities (Mitte 40%, Prenzlauer Berg 30%, Friedrichshain-Kreuzberg 30%) to reflect realistic demand concentration in central Berlin.

**Status Distribution:**
Hardcodes status probabilities to mimic fleet behavior: 60% of logs track live active trips (`Passengers_Onboard`), while 40% track fleet waste via `Idle` and `En_Route` indicators.

#### Data Cleansing Limits
**Gaussian Curves:**
Ride durations and fare amounts follow Gaussian (normal) distributions. The typical ride is centered around 15 minutes, producing an expected fare of approximately €22.00 (€4.00 base + €1.20/min × 15 mins).

**Boundary Truncation:**
Outliers are a common data quality issue. The `.clip()` function guarantees that no mathematically random variations result in negative trips or impossible metrics. No trip can fall below 3 minutes / €6.00, or exceed 45 minutes / €60.00.

---

#### Data Generation — Trip Duration

**`trip_duration_mins`** is stored as a nullable integer (`Int64`).

Trip durations are rounded to whole minutes rather than stored as decimals. This is intentional: the analytical focus of this project is fleet-level operational leakage and supply-demand mismatch, which are measured in aggregated time windows — not per-second precision. Integer minutes are standard in ride-sharing datasets (Uber, Lyft, BVG) and produce cleaner SQL aggregations without any meaningful loss of insight.

Rows where `driver_status` is `Idle` or `En Route` equal to 0, as no active trip is in progress. This distinction is load-bearing for leakage detection queries.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
# This ensures that every time the script runs.
# Generates the exact same dataset, which is crucial for testing and debugging SQL transformations later.
np.random.seed(42)

               

# Configuration
num_trips = 5000
districts = ['Mitte', 'Prenzlauer Berg', 'Friedrichshain-Kreuzberg']
statuses = ['Idle', 'Passengers_Onboard', 'En Route']

# Generate Timestamps over a 3-week period (21 days) leading up to today
end_date = datetime.now()
start_date = end_date - timedelta(days=21)

# Generate random minutes within a 21-day window
random_minutes = np.random.randint(0, 21 * 24 * 60, size=num_trips)
timestamps = [start_date + timedelta(minutes=int(m)) for m in random_minutes]

# Generate District Distribution (weighted slightly toward Mitte)
pickup_districts = np.random.choice(districts, size=num_trips, p=[0.4, 0.3, 0.3])

# Generate IDs
trip_ids = [f"TRIP_{10000 + i}" for i in range(num_trips)]
driver_ids = [f"DRV_{np.random.randint(100, 150)}" for _ in range(num_trips)]

# Generate Driver Statuses
# If they have a fare, status is Passenger_Onboard. 
# Otherwise, they are Idle or En_Route.
driver_statuses = np.random.choice(statuses, size=num_trips, p=[0.2, 0.6, 0.2])

# Generate realistic durations first (Centered around 15 minutes)
trip_durations_mins = np.random.normal(loc=15, scale=5, size=num_trips).round(1)
trip_durations_mins = np.clip(trip_durations_mins, 3, 45).round().astype(int)

# Calculate fare based on duration (e.g., €4.00 base price + €1.20 per minute)
# This ensures that a longer trip always equals a higher fare
# the last variable represents real-world randomness (traffic light, contruction...)
base_fare = 4.00
rate_per_minute = 1.20
random_traffic_variation = np.random.normal(loc=0, scale=2, size=num_trips) # adds slight unpredictability

# Takes a fixed base fare, adds a per-minute tariff driven by the trip's duration, 
# Introduces a normal distribution variation to simulate real-world traffic unpredictability. 
# Rounds the result to two decimal places to match standard Euro currency formatting."
fare_amounts_euros = (base_fare + (trip_durations_mins * rate_per_minute) + random_traffic_variation).round(2)
fare_amounts_euros = np.clip(fare_amounts_euros, 6.00, 60.00)


# Convert numeric duration to MM:SS format
def to_mmss(val):
    if np.isnan(val):
        return None
    total_seconds = int(round(val * 60))
    mm = total_seconds // 60
    ss = total_seconds % 60
    return f"{mm:02d}:{ss:02d}"

# Create the DataFrame
df = pd.DataFrame({
    'trip_id': trip_ids,
    'timestamp': timestamps,
    'pickup_district': pickup_districts,
    'driver_id': driver_ids,
    'driver_status': driver_statuses,
    'trip_duration_mins': trip_durations_mins,
    'fare_amount_euros': fare_amounts_euros
})

# Zero out duration and fare for Idle and En Route rows — mask on DataFrame directly
df.loc[df['driver_status'] != 'Passengers_Onboard', 'trip_duration_mins'] = 0
df.loc[df['driver_status'] != 'Passengers_Onboard', 'fare_amount_euros'] = 0.00

# Enforce clean types for BigQuery
df['trip_duration_mins'] = df['trip_duration_mins'].astype(int)
df['fare_amount_euros'] = df['fare_amount_euros'].astype(float).round(2)


# Sanity check before export
print("Status distribution:")
print(df['driver_status'].value_counts())
print("\nNull check:")
print(df.isnull().sum())
print("\nSample of Idle/En Route rows:")
print(df[df['driver_status'] != 'Passengers_Onboard'][['driver_status', 'trip_duration_mins', 'fare_amount_euros']].head())


# Sort by timestamp so it looks like a real database log sequential stream
df = df.sort_values(by='timestamp').reset_index(drop=True)

# Export to CSV

output_filename = 'berlin_taxi_trips_raw.csv'
df.to_csv(output_filename, index=False)

print(f"Generated {len(df)} realistic rows across 3 weeks -> 'berlin_taxi_trips_raw.csv'")
df